# ALIGN-PARETO — M2: SFT (+ DPO) from Qwen3-0.6B-Base
Builds the λ-controllable model from the **unaligned** pretrained model (spec §18–19):

| Stage | Data | Output | Role |
|---|---|---|---|
| 1. SFT | human-written replies: no_robots (~8.6k) + oasst2 best-ranked English (6k), refusal-like replies removed | `models/m2/sft_adapter` | helpful-only chat model, **λ = 0** |
| 2. DPO | hh-rlhf preference pairs (3.5k harmless + 1.5k helpful) | `models/m2/dpo_adapter` | safety alignment, **λ = 1** |

`STAGES` below picks which stages run (`"sft"` = λ = 0 only; `"dpo"` alone reuses a saved SFT adapter).

**Kaggle setup:** input `align-pareto-src` attached (must contain `data/eval/eval_prompts.csv`), **GPU T4**, internet on
(the model and datasets come from the Hugging Face Hub; all public). Run from this machine with
`kaggle/run_on_kaggle.sh m2`. All logic lives in `src/`; this notebook only runs it and shows the results.

In [ ]:
# 1. Environment (one GPU: the trainers would otherwise wrap the model in DataParallel)
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, json, platform, shutil, glob, subprocess
from pathlib import Path
import importlib.metadata as md

print("Python", platform.python_version())
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU: M2 needs one (Settings -> Accelerator -> GPU T4)")

In [ ]:
# 2. Dependencies (pinned to the versions the code was tested with)
PINNED = ["transformers==5.22.2", "trl==1.17.1", "peft==0.21.2", "datasets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)
# Kaggle preinstalls torchao 0.10; peft 0.21 refuses to wrap layers while an old torchao is importable (unused here)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "huggingface_hub"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} MISSING")

In [ ]:
# 3. Project code: Kaggle dataset `align-pareto-src` -> /kaggle/working/align-pareto; locally, the repo itself
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/src/alignment/train_sft.py", recursive=True)
    assert hits, "align-pareto-src is missing or out of date (no src/alignment/train_sft.py): attach its latest version."
    ROOT = Path("/kaggle/working/align-pareto")
    shutil.copytree(Path(hits[0]).parents[2], ROOT, dirs_exist_ok=True)
else:
    ROOT = Path.cwd().parent
assert (ROOT / "data/eval/eval_prompts.csv").exists(), "data/eval/eval_prompts.csv missing: attach the latest align-pareto-src"
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# 4. Run M2 (QUICK=True: 3-step smoke test)
QUICK = False
STAGES = ("sft",)              # ("sft",) = lambda = 0 only; ("sft", "dpo") = both
from src.run_m2 import run_m2
metrics = run_m2(ROOT, quick=QUICK, stages=STAGES)
TAG = "m2_quick" if QUICK else "m2"
RES = ROOT / "results" / TAG
if ON_KAGGLE:  # keep the output small: raw downloads and optimizer checkpoints are not needed afterwards
    shutil.rmtree(ROOT / "data/raw/hf_cache", ignore_errors=True)
    for ckpt in (ROOT / "models" / TAG).glob("*_checkpoints"):
        shutil.rmtree(ckpt, ignore_errors=True)

In [ ]:
# 5. Training checks: SFT loss falls; DPO loss falls and reward accuracy rises above 0.5; adapters reload
print(json.dumps({k: metrics[k] for k in ["sft", "dpo", "reload_check"] if k in metrics}, indent=2))
from IPython.display import Image, display
for name in ["sft_training.png", "dpo_training.png"]:
    if (RES / "plots" / name).exists():
        display(Image(filename=str(RES / "plots" / name), width=900))

In [ ]:
# 6. Smoke test: raw base model vs lambda = 0 (SFT) [vs lambda = 1 (SFT + DPO)], sampled with a fixed seed
import pandas as pd
pd.set_option("display.max_colwidth", 300)
smoke = pd.read_csv(RES / "smoke_test.csv")
print(metrics["smoke_test"])
display(smoke[[c for c in ["quadrant", "prompt", "reply_base_model", "reply_lambda_0", "reply_lambda_1"] if c in smoke]])

In [ ]:
# 7. One zip with everything M3 needs, for download from the Output panel
if ON_KAGGLE:
    adapters = " ".join(f"models/{TAG}/{a}" for a in ["sft_adapter", "dpo_adapter"] if (ROOT / "models" / TAG / a).exists())
    subprocess.run(f"cd {ROOT} && zip -qr /kaggle/working/m2_outputs.zip results/{TAG} {adapters}", shell=True, check=True)
    print("Saved /kaggle/working/m2_outputs.zip")

## Next: M3
The λ sweep loads `models.load_aligned_model(sft_adapter, dpo_adapter)` and scales the DPO adapter with
`adapters.set_peft_lora_lambda`.